In [ ]:
%write_file pinet_rmd17.py

import molpot as mpot
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cuda"
)
rmd17_ds.add_process(mpot.process.NeighborList(cutoff=5.0))

train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.95, .05])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=10,
    shuffle=False,
)

pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    [64, 1],
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
f_readout = mpot.potential.nnp.readout.DPairPot(
    fx_key=("predicts", "energy"),
    dx_key=("pairs", "dist"),
    to_key=("predicts", "forces"),
    create_graph=True
)
model = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

save_dir = Path("pinet2-rmd17")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
loss_fn = mpot.engine.MultiTargetLoss(
    torch.nn.MSELoss(), [("energy", "energy", 1.0), ("forces", "forces", 10.0)]
)

potential_trainer = mpot.PotentialTrainer(
    model = model,
    optimizer = scheduler,
    loss_fn = loss_fn,
    device = "cpu",
    no_grad_eval = True,
)
potential_trainer.add_lr_scheduler()
potential_trainer.add_checkpoint(save_dir)
potential_trainer.add_metric("e_mae", MeanAbsoluteError(lambda pred, label: (pred["energy"], label["energy"])), BatchWise(), target="all")
potential_trainer.add_metric("f_mae", MeanAbsoluteError(lambda pred, label: (pred["forces"], label["forces"])), BatchWise(), target="all")
potential_trainer.attach_tensorboard(
    save_dir/"tb",
)
state = potential_trainer.run(train_data=train_dl, max_steps=1000, eval_data=eval_dl)